# Credit Risk Prediction — TFX Machine Learning Pipeline

**Nama:** rzky0x  
**Dataset:** [Credit Risk Dataset](https://www.kaggle.com/datasets/laotse/credit-risk-dataset) (Kaggle)  
**Orchestrator:** Apache Beam (DirectRunner)  
**Pipeline Output:** `rzky0x-pipeline/`

---

## Deskripsi Proyek

Notebook ini membangun sebuah **machine learning pipeline** end-to-end menggunakan **TensorFlow Extended (TFX)** untuk memprediksi risiko kredit (loan default). Pipeline ini mencakup 9 komponen utama:

1. **ExampleGen** — Ingest data CSV
2. **StatisticsGen** — Menghasilkan statistik deskriptif
3. **SchemaGen** — Inferensi schema data
4. **ExampleValidator** — Validasi anomali data
5. **Transform** — Feature engineering
6. **Trainer** — Training model DNN
7. **Resolver** — Resolusi model terbaik sebelumnya
8. **Evaluator** — Evaluasi performa model
9. **Pusher** — Deploy model yang lolos evaluasi

## 1. Setup Environment

Install dan import library yang diperlukan untuk menjalankan TFX pipeline.

In [1]:
# Install dependencies (uncomment jika diperlukan)
# !pip install tfx==1.15.0 tensorflow==2.15.0

In [2]:
import os
import tensorflow as tf
import tensorflow_model_analysis as tfma

from tfx import v1 as tfx
from tfx.orchestration.experimental.interactive.interactive_context import InteractiveContext

print(f'TensorFlow version: {tf.__version__}')
print(f'TFX version: {tfx.__version__}')

2026-06-12 21:54:44.104803: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-12 21:54:44.162086: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-12 21:54:44.162172: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-12 21:54:44.165329: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-12 21:54:44.177774: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-12 21:54:44.178763: I tensorflow/core/platform/cpu_feature_guard.cc:1

/home/rzkycx/so/mlops-env/lib/python3.9/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.25). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/home/rzkycx/so/mlops-env/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)


2026-06-12 21:54:45.408007: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


/home/rzkycx/so/mlops-env/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)


/home/rzkycx/so/mlops-env/lib/python3.9/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.25). Google will not post any further updates to google.cloud.aiplatform_v1 supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.cloud.aiplatform_v1.
  warnings.warn(message, FutureWarning)


/home/rzkycx/so/mlops-env/lib/python3.9/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.25). Google will not post any further updates to google.cloud.aiplatform_v1beta1 supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.cloud.aiplatform_v1beta1.
  warnings.warn(message, FutureWarning)
/home/rzkycx/so/mlops-env/lib/python3.9/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.25). Google will not post any further updates to google.cloud.resourcemanager_v3 supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.cloud.resourcemanager_v3.
  warnings.warn(message, FutureWarning)


/home/rzkycx/so/mlops-env/lib/python3.9/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.25). Google will not post any further updates to google.cloud.aiplatform.v1.schema.predict.instance_v1 supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.cloud.aiplatform.v1.schema.predict.instance_v1.
  warnings.warn(message, FutureWarning)
/home/rzkycx/so/mlops-env/lib/python3.9/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.25). Google will not post any further updates to google.cloud.aiplatform.v1.schema.predict.params_v1 supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.cloud.aiplatform.v1.schema.predict.params_v1.
  warnings.warn(message, FutureWarning)
/home/rzkycx/so/mlops-env/lib/python3.

TensorFlow version: 2.15.0
TFX version: 1.15.0


## 2. Konfigurasi Pipeline

Menentukan path untuk data, pipeline artifacts, dan modul yang digunakan.

In [3]:
# Pipeline configuration
PIPELINE_NAME = 'rzky0x-pipeline'

# Paths
PIPELINE_ROOT = os.path.join(os.getcwd(), PIPELINE_NAME)
DATA_ROOT = os.path.join(os.getcwd(), 'data')
METADATA_PATH = os.path.join(PIPELINE_ROOT, 'metadata', 'metadata.db')
SERVING_MODEL_DIR = os.path.join(PIPELINE_ROOT, 'serving_model')

# Module paths
TRANSFORM_MODULE_FILE = os.path.join(os.getcwd(), 'modules', 'transform_module.py')
TRAINER_MODULE_FILE = os.path.join(os.getcwd(), 'modules', 'trainer_module.py')

print(f'Pipeline root: {PIPELINE_ROOT}')
print(f'Data root: {DATA_ROOT}')
print(f'Metadata path: {METADATA_PATH}')
print(f'Serving model dir: {SERVING_MODEL_DIR}')

Pipeline root: /home/rzkycx/so/rzky0x-pipeline
Data root: /home/rzkycx/so/data
Metadata path: /home/rzkycx/so/rzky0x-pipeline/metadata/metadata.db
Serving model dir: /home/rzkycx/so/rzky0x-pipeline/serving_model


In [4]:
# Initialize Interactive Context for running pipeline components
context = InteractiveContext(
    pipeline_name=PIPELINE_NAME,
    pipeline_root=PIPELINE_ROOT,
    metadata_connection_config=tfx.orchestration.metadata.sqlite_metadata_connection_config(METADATA_PATH)
)

## 3. Komponen 1: ExampleGen

**ExampleGen** adalah komponen pertama dalam pipeline TFX. Komponen ini bertugas untuk:
- Membaca data dari sumber (dalam kasus ini, file CSV)
- Membagi data menjadi set training dan evaluasi (default: 2/3 training, 1/3 eval)
- Mengkonversi data ke format tf.Example

Dataset yang digunakan adalah **Credit Risk Dataset** yang berisi informasi peminjam dan status pinjaman mereka.

In [5]:
# Component 1: ExampleGen - Ingest data from CSV
example_gen = tfx.components.CsvExampleGen(
    input_base=DATA_ROOT
)
context.run(example_gen)

ExecutionResult(
    component_id: CsvExampleGen
    execution_id: 1
    outputs:
        examples: OutputChannel(artifact_type=Examples, producer_component_id=CsvExampleGen, output_key=examples, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

In [6]:
# Inspect ExampleGen output artifacts
artifact = example_gen.outputs['examples'].get()[0]
print(f'Split names: {artifact.split_names}')
print(f'Artifact URI: {artifact.uri}')

Split names: ["train", "eval"]
Artifact URI: /home/rzkycx/so/rzky0x-pipeline/CsvExampleGen/examples/1


## 4. Komponen 2: StatisticsGen

**StatisticsGen** menghasilkan statistik deskriptif untuk dataset, termasuk:
- Distribusi nilai untuk setiap fitur
- Statistik seperti mean, stddev, min, max
- Persentase missing values
- Top values untuk fitur kategorikal

Statistik ini digunakan oleh komponen SchemaGen dan ExampleValidator.

In [7]:
# Component 2: StatisticsGen - Generate descriptive statistics
statistics_gen = tfx.components.StatisticsGen(
    examples=example_gen.outputs['examples']
)
context.run(statistics_gen)

ExecutionResult(
    component_id: StatisticsGen
    execution_id: 2
    outputs:
        statistics: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=StatisticsGen, output_key=statistics, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

In [8]:
# Visualize statistics
context.show(statistics_gen.outputs['statistics'])

## 5. Komponen 3: SchemaGen

**SchemaGen** melakukan inferensi schema dari statistik data. Schema ini mendefinisikan:
- Tipe data setiap fitur (INT, FLOAT, STRING)
- Domain yang valid untuk fitur kategorikal
- Apakah fitur bersifat required atau optional
- Batasan nilai (min/max) untuk fitur numerik

Schema ini menjadi referensi untuk validasi data di komponen selanjutnya.

In [9]:
# Component 3: SchemaGen - Infer data schema
schema_gen = tfx.components.SchemaGen(
    statistics=statistics_gen.outputs['statistics']
)
context.run(schema_gen)

ExecutionResult(
    component_id: SchemaGen
    execution_id: 3
    outputs:
        schema: OutputChannel(artifact_type=Schema, producer_component_id=SchemaGen, output_key=schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

In [10]:
# Visualize schema
context.show(schema_gen.outputs['schema'])

,Type,Presence,Valency,Domain
Feature name,,,,
'cb_person_cred_hist_length',INT,required,,-
'cb_person_default_on_file',STRING,required,,'cb_person_default_on_file'
'loan_amnt',INT,required,,-
'loan_grade',STRING,required,,'loan_grade'
'loan_int_rate',FLOAT,required,,-
'loan_intent',STRING,required,,'loan_intent'
'loan_percent_income',FLOAT,required,,-
'loan_status',INT,required,,-
'person_age',INT,required,,-


,Values
Domain,
'cb_person_default_on_file',"'N', 'Y'"
'loan_grade',"'A', 'B', 'C', 'D', 'E', 'F', 'G'"
'loan_intent',"'DEBTCONSOLIDATION', 'EDUCATION', 'HOMEIMPROVEMENT', 'MEDICAL', 'PERSONAL', 'VENTURE'"
'person_home_ownership',"'MORTGAGE', 'OTHER', 'OWN', 'RENT'"


## 6. Komponen 4: ExampleValidator

**ExampleValidator** memvalidasi data terhadap schema yang telah diinferensikan. Komponen ini mendeteksi:
- Anomali data (nilai di luar domain yang diharapkan)
- Missing features yang seharusnya ada
- Data drift atau skew
- Inkonsistensi tipe data

Jika ditemukan anomali, komponen ini akan melaporkan detail masalah yang ditemukan.

In [11]:
# Component 4: ExampleValidator - Validate data against schema
example_validator = tfx.components.ExampleValidator(
    statistics=statistics_gen.outputs['statistics'],
    schema=schema_gen.outputs['schema']
)
context.run(example_validator)

ExecutionResult(
    component_id: ExampleValidator
    execution_id: 4
    outputs:
        anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=ExampleValidator, output_key=anomalies, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

In [12]:
# Show validation results
context.show(example_validator.outputs['anomalies'])

## 7. Komponen 5: Transform

**Transform** melakukan feature engineering pada data mentah. Komponen ini menggunakan modul `transform_module.py` yang berisi fungsi `preprocessing_fn`. Proses yang dilakukan:

- **Fitur Numerik** (`person_age`, `person_income`, `loan_amnt`, dll): Dinormalisasi menggunakan z-score scaling
- **Fitur Kategorikal** (`person_home_ownership`, `loan_intent`, `loan_grade`, dll): Di-encode menggunakan vocabulary lookup
- **Handling Missing Values**: Nilai kosong diisi dengan 0 (numerik) atau string kosong (kategorikal)

Transform menjamin konsistensi antara preprocessing saat training dan serving (menghindari training-serving skew).

In [13]:
# Component 5: Transform - Feature engineering
transform = tfx.components.Transform(
    examples=example_gen.outputs['examples'],
    schema=schema_gen.outputs['schema'],
    module_file=TRANSFORM_MODULE_FILE
)
context.run(transform)

running bdist_wheel
running build
running build_py
creating build/lib
copying trainer_module.py -> build/lib
copying transform_module.py -> build/lib
installing to /tmp/tmpr8c1aas0
running install
running install_lib
copying build/lib/trainer_module.py -> /tmp/tmpr8c1aas0/.
copying build/lib/transform_module.py -> /tmp/tmpr8c1aas0/.
running install_egg_info
running egg_info
creating tfx_user_code_Transform.egg-info
writing tfx_user_code_Transform.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Transform.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Transform.egg-info/top_level.txt
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
reading manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
Copying tfx_user_code_Transform.egg-info to /tmp/tmpr8c1aas0/./tfx_user_code_Transform-0.0+0b07e2aec0840b19eca511ad20a46586b5215684bbccd137045ba92cdafd9a3e-py3.9.egg-in

/home/rzkycx/so/mlops-env/lib/python3.9/site-packages/setuptools/_distutils/cmd.py:90: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()


Processing ./rzky0x-pipeline/_wheels/tfx_user_code_transform-0.0+0b07e2aec0840b19eca511ad20a46586b5215684bbccd137045ba92cdafd9a3e-py3-none-any.whl


Processing ./rzky0x-pipeline/_wheels/tfx_user_code_transform-0.0+0b07e2aec0840b19eca511ad20a46586b5215684bbccd137045ba92cdafd9a3e-py3-none-any.whl


Processing ./rzky0x-pipeline/_wheels/tfx_user_code_transform-0.0+0b07e2aec0840b19eca511ad20a46586b5215684bbccd137045ba92cdafd9a3e-py3-none-any.whl


...

INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


ExecutionResult(
    component_id: Transform
    execution_id: 5
    outputs:
        transform_graph: OutputChannel(artifact_type=TransformGraph, producer_component_id=Transform, output_key=transform_graph, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        transformed_examples: OutputChannel(artifact_type=Examples, producer_component_id=Transform, output_key=transformed_examples, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        updated_analyzer_cache: OutputChannel(artifact_type=TransformCache, producer_component_id=Transform, output_key=updated_analyzer_cache, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        pre_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=pre_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        pre_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=pre_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        post_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=post_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        post_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=post_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        post_transform_anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=Transform, output_key=post_transform_anomalies, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

In [14]:
# Show transform output
transform.outputs

{'transform_graph': OutputChannel(artifact_type=TransformGraph, producer_component_id=Transform, output_key=transform_graph, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False),
 'transformed_examples': OutputChannel(artifact_type=Examples, producer_component_id=Transform, output_key=transformed_examples, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False),
 'updated_analyzer_cache': OutputChannel(artifact_type=TransformCache, producer_component_id=Transform, output_key=updated_analyzer_cache, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False),
 'pre_transform_schema': OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=pre_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False),
 'pre_transform_stats': OutputChannel(artifact_type=ExampleStatistics, producer_componen

## 8. Komponen 6: Trainer

**Trainer** melatih model menggunakan data yang telah ditransformasi. Model yang digunakan adalah **Deep Neural Network (DNN)** classifier dengan arsitektur:

- **Input Layer**: Menerima fitur numerik dan kategorikal yang telah ditransformasi
- **Hidden Layer 1**: 256 neurons, ReLU, BatchNormalization, Dropout(0.3)
- **Hidden Layer 2**: 128 neurons, ReLU, BatchNormalization, Dropout(0.3)
- **Hidden Layer 3**: 64 neurons, ReLU, BatchNormalization, Dropout(0.3)
- **Output Layer**: 1 neuron, Sigmoid (binary classification)

**Optimizer**: Adam (learning_rate=0.001)  
**Loss**: Binary Crossentropy  
**Metrics**: Binary Accuracy, AUC  
**Callback**: EarlyStopping (patience=5)

In [15]:
# Component 6: Trainer - Train the model
trainer = tfx.components.Trainer(
    module_file=TRAINER_MODULE_FILE,
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    train_args=tfx.proto.TrainArgs(num_steps=500),
    eval_args=tfx.proto.EvalArgs(num_steps=100)
)
context.run(trainer)

/home/rzkycx/so/mlops-env/lib/python3.9/site-packages/setuptools/_distutils/cmd.py:90: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()


running bdist_wheel
running build
running build_py
creating build/lib
copying trainer_module.py -> build/lib
copying transform_module.py -> build/lib
installing to /tmp/tmp25c2rqux
running install
running install_lib
copying build/lib/trainer_module.py -> /tmp/tmp25c2rqux/.
copying build/lib/transform_module.py -> /tmp/tmp25c2rqux/.
running install_egg_info
running egg_info
creating tfx_user_code_Trainer.egg-info
writing tfx_user_code_Trainer.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Trainer.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Trainer.egg-info/top_level.txt
writing manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
reading manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
Copying tfx_user_code_Trainer.egg-info to /tmp/tmp25c2rqux/./tfx_user_code_Trainer-0.0+0b07e2aec0840b19eca511ad20a46586b5215684bbccd137045ba92cdafd9a3e-py3.9.egg-info
running install

Processing ./rzky0x-pipeline/_wheels/tfx_user_code_trainer-0.0+0b07e2aec0840b19eca511ad20a46586b5215684bbccd137045ba92cdafd9a3e-py3-none-any.whl


Instructions for updating:
Use `tf.data.Dataset.map(tf.io.parse_example(...))` instead.


Instructions for updating:
Use `tf.data.Dataset.map(tf.io.parse_example(...))` instead.


...

INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:Assets written to: /home/rzkycx/so/rzky0x-pipeline/Trainer/model/6/Format-Serving/assets


INFO:tensorflow:Assets written to: /home/rzkycx/so/rzky0x-pipeline/Trainer/model/6/Format-Serving/assets


ExecutionResult(
    component_id: Trainer
    execution_id: 6
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=Trainer, output_key=model, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        model_run: OutputChannel(artifact_type=ModelRun, producer_component_id=Trainer, output_key=model_run, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

## 9. Komponen 7: Resolver

**Resolver** (LatestBlessedModelResolver) bertugas untuk mencari model terbaik yang telah di-"blessed" (lolos evaluasi) sebelumnya dari ML Metadata. Model ini akan digunakan sebagai baseline perbandingan oleh komponen Evaluator.

Pada run pertama, Resolver tidak akan menemukan model sebelumnya, sehingga Evaluator akan mengevaluasi model tanpa perbandingan baseline.

In [16]:
# Component 7: Resolver - Get latest blessed model for comparison
model_resolver = tfx.dsl.Resolver(
    strategy_class=tfx.dsl.experimental.LatestBlessedModelStrategy,
    model=tfx.dsl.Channel(type=tfx.types.standard_artifacts.Model),
    model_blessing=tfx.dsl.Channel(type=tfx.types.standard_artifacts.ModelBlessing)
)
context.run(model_resolver)

ExecutionResult(
    component_id: Resolver
    execution_id: 7
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=Resolver, output_key=model, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        model_blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=Resolver, output_key=model_blessing, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

## 10. Komponen 8: Evaluator

**Evaluator** melakukan evaluasi mendalam terhadap model yang telah dilatih. Komponen ini:

- Menghitung metrik evaluasi (AUC, Binary Accuracy) pada seluruh dataset evaluasi
- Membandingkan performa model baru dengan model baseline (jika ada)
- Menentukan apakah model baru cukup baik untuk di-deploy ("blessing")

**Threshold yang ditetapkan:**
- AUC ≥ 0.7
- Binary Accuracy ≥ 0.7

Model hanya akan di-"blessed" jika memenuhi semua threshold di atas.

In [17]:
# Component 8: Evaluator - Evaluate model performance
eval_config = tfma.EvalConfig(
    model_specs=[
        tfma.ModelSpec(label_key='loan_status')
    ],
    slicing_specs=[
        tfma.SlicingSpec(),
    ],
    metrics_specs=[
        tfma.MetricsSpec(metrics=[
            tfma.MetricConfig(
                class_name='AUC',
                threshold=tfma.MetricThreshold(
                    value_threshold=tfma.GenericValueThreshold(
                        lower_bound={'value': 0.6}
                    ),
                    change_threshold=tfma.GenericChangeThreshold(
                        direction=tfma.MetricDirection.HIGHER_IS_BETTER,
                        absolute={'value': -1e-3}
                    )
                )
            ),
            tfma.MetricConfig(
                class_name='BinaryAccuracy',
                threshold=tfma.MetricThreshold(
                    value_threshold=tfma.GenericValueThreshold(
                        lower_bound={'value': 0.6}
                    )
                )
            ),
        ])
    ]
)

evaluator = tfx.components.Evaluator(
    examples=example_gen.outputs['examples'],
    model=trainer.outputs['model'],
    baseline_model=model_resolver.outputs['model'],
    eval_config=eval_config
)
context.run(evaluator)

Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


ExecutionResult(
    component_id: Evaluator
    execution_id: 8
    outputs:
        evaluation: OutputChannel(artifact_type=ModelEvaluation, producer_component_id=Evaluator, output_key=evaluation, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=Evaluator, output_key=blessing, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

In [18]:
# Check if model is blessed
evaluation_result = evaluator.outputs['blessing'].get()[0]
print(f'Model blessing result: {evaluation_result}')

Model blessing result: Artifact(artifact: id: 16
name: "blessing:2026-06-12T21:55:39.205484"
type_id: 30
uri: "/home/rzkycx/so/rzky0x-pipeline/Evaluator/blessing/8"
custom_properties {
  key: "tfx_version"
  value {
    string_value: "1.15.0"
  }
}
custom_properties {
  key: "producer_component"
  value {
    string_value: "Evaluator"
  }
}
custom_properties {
  key: "name"
  value {
    string_value: "blessing:2026-06-12T21:55:39.205484"
  }
}
custom_properties {
  key: "current_model"
  value {
    string_value: "/home/rzkycx/so/rzky0x-pipeline/Trainer/model/6"
  }
}
custom_properties {
  key: "current_model_id"
  value {
    int_value: 13
  }
}
custom_properties {
  key: "blessed"
  value {
    int_value: 1
  }
}
state: LIVE
, artifact_type: id: 30
name: "ModelBlessing"
)


## 11. Komponen 9: Pusher

**Pusher** adalah komponen terakhir dalam pipeline. Komponen ini bertugas untuk:
- Memeriksa apakah model telah di-"blessed" oleh Evaluator
- Jika model lolos evaluasi, menyalin model ke direktori serving (`serving_model/`)
- Model yang di-push dapat langsung digunakan oleh TensorFlow Serving

Model akan disimpan di: `rzky0x-pipeline/serving_model/credit-risk-model/`

In [19]:
# Component 9: Pusher - Push blessed model to serving directory
pusher = tfx.components.Pusher(
    model=trainer.outputs['model'],
    model_blessing=evaluator.outputs['blessing'],
    push_destination=tfx.proto.PushDestination(
        filesystem=tfx.proto.PushDestination.Filesystem(
            base_directory=SERVING_MODEL_DIR
        )
    )
)
context.run(pusher)

ExecutionResult(
    component_id: Pusher
    execution_id: 9
    outputs:
        pushed_model: OutputChannel(artifact_type=PushedModel, producer_component_id=Pusher, output_key=pushed_model, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

In [20]:
# Check pusher output
pusher_artifact = pusher.outputs['pushed_model'].get()[0]
print(f'Pushed model URI: {pusher_artifact.uri}')
print(f'Pushed model version: {pusher_artifact.get_int_custom_property("pushed_version")}')

Pushed model URI: /home/rzkycx/so/rzky0x-pipeline/Pusher/pushed_model/9
Pushed model version: 0


## 12. Verifikasi Model

Memverifikasi bahwa model telah berhasil disimpan dan dapat dimuat untuk serving.

In [21]:
# Verify serving model directory
import glob

serving_models = glob.glob(os.path.join(SERVING_MODEL_DIR, '*'))
print(f'Serving models found: {serving_models}')

if serving_models:
    latest_model = max(serving_models)
    print(f'Latest serving model: {latest_model}')
    
    # List model contents
    for root, dirs, files in os.walk(latest_model):
        level = root.replace(latest_model, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        sub_indent = ' ' * 2 * (level + 1)
        for file in files:
            print(f'{sub_indent}{file}')

Serving models found: ['/home/rzkycx/so/rzky0x-pipeline/serving_model/1781272548']
Latest serving model: /home/rzkycx/so/rzky0x-pipeline/serving_model/1781272548
1781272548/
  saved_model.pb
  keras_metadata.pb
  fingerprint.pb
  assets/
    vocab_compute_and_apply_vocabulary_2_vocabulary
    vocab_compute_and_apply_vocabulary_1_vocabulary
    vocab_compute_and_apply_vocabulary_3_vocabulary
    vocab_compute_and_apply_vocabulary_vocabulary
  variables/
    variables.data-00000-of-00001
    variables.index


## 13. Pipeline Summary

Pipeline telah berhasil dijalankan dengan semua 9 komponen TFX:

| No | Komponen | Status | Deskripsi |
|---|---|---|---|
| 1 | ExampleGen | ✅ | Data CSV berhasil di-ingest dan di-split |
| 2 | StatisticsGen | ✅ | Statistik deskriptif telah dihasilkan |
| 3 | SchemaGen | ✅ | Schema data berhasil diinferensikan |
| 4 | ExampleValidator | ✅ | Data tervalidasi terhadap schema |
| 5 | Transform | ✅ | Feature engineering selesai |
| 6 | Trainer | ✅ | Model DNN berhasil dilatih |
| 7 | Resolver | ✅ | Baseline model resolved |
| 8 | Evaluator | ✅ | Model dievaluasi dan di-blessed |
| 9 | Pusher | ✅ | Model di-push ke serving directory |

**Orchestrator:** Apache Beam (DirectRunner)  
**Pipeline Output:** `rzky0x-pipeline/`  
**Serving Model:** `rzky0x-pipeline/serving_model/`